In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import keyring
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# User set the main directory using keyring.  For example:
#    import keyring
#    keyring.set_password('msp', 'vmt_reduction_dir', 'path to the desired directory')
# This will be saved on your local machine

teams_dir = keyring.get_password('msp', 'vmt_reduction_dir')

In [ ]:
df = pd.read_csv(teams_dir + "/data_processed/tbi_cleaned.csv")
df

In [ ]:
MILES_PER_METER = 0.000621371

## Car Analysis

In [ ]:
car = pd.read_parquet(teams_dir + "/Data_processed/geodata/car_congestion_nogeom.parquet")
car

In [ ]:
def get_car_data(row, field):
    hour = int(row["arrive_time"][0:2])
    sunday = row["travel_dow"] == "Sunday"
    saturday = row["travel_dow"] == "Saturday"
    if sunday:
        query = "sundays"
    elif saturday:
        query = "saturdays_"
    else:
        query = "weekdays_"
    
    if hour >= 0 and hour <= 5:
        query += "0-6"
    elif hour >= 20 and hour <= 23:
        query += "20-24"
    else:
        query += str(hour) + "-" + str(hour + 1)

    if (query, row["trip_id"]) not in car.index:
        print(row["trip_id"])
        return np.nan

    return car.loc[(query, row["trip_id"])][field]

In [ ]:
df["rerouting_duration_car"] = df.apply(lambda x: get_car_data(x, "duration_seconds"), axis=1) / 60
df["rerouting_distance_car"] = df.apply(lambda x: get_car_data(x, "distance_meters"), axis=1) * MILES_PER_METER

Comparing the summary statistics, it seems that rerouting durations genreally seem to underestmate trip durations in nearly all indicators

In [ ]:
pd.concat([df[df["mode"] == "Car"]["rerouting_duration_car"].describe(), df[df["mode"] == "Car"]["duration"].describe()], axis=1)

Rerouting analysis seems to generally underestmiate trip durations, with most linked trips being below the y=x line in the scatter plot below. This could be due to implicit bias in the sampled durations in the TBI data, with self-reported/estimated durations not being representative of the actual trip. This could also be due to the cleaning that was done to clean duration values (and the lack of cleaning in some cases, as seen with the rightmost outliers).

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))
sns.scatterplot(data=df[df["mode"] == "Car"], x="duration", y="rerouting_duration_car", ax=ax)
x_ref = np.linspace(0, 150)
plt.plot(x_ref, x_ref, color="r", linewidth=2)

The summary statistics mirror the graph, with about 75% of all linked trips having greater actual durations than rerouting durations. It should be noted that for 90% of trips, the absolute difference is less than 10 minutes, which is relatively negligible.

In [ ]:
(df[df["mode"] == "Car"]["duration"] - df[df["mode"] == "Car"]["rerouting_duration_car"]).describe(percentiles=[0.05, 0.1, 0.15, 0.2, 0.25, 0.5, 0.75, 0.8, 0.85, 0.9, 0.95])

This scatter plot outlines the difference between duration of non-car modes and their corresponding rerouted car durations. As is to be expected, for the vast majority of trips, going by car beats going by the non-car mode (most of the points are below the y=x line).

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))
sns.scatterplot(data=df[df["mode"] != "Car"], x="duration", y="rerouting_duration_car", ax=ax)
x_ref = np.linspace(0, 150)
plt.plot(x_ref, x_ref, color="r", linewidth=2)

This is once again mirrored by the summary statistics, with only about 5% of all linked non-car trips having parity/being faster than their corresponding rerouted car duration.

In [ ]:
(df[df["mode"] != "Car"]["duration"] - df[df["mode"] != "Car"]["rerouting_duration_car"]).describe(percentiles=[0.05, 0.1, 0.15, 0.2, 0.25, 0.5, 0.75, 0.8, 0.85, 0.9, 0.95])

Unlike with duration, rerouting distances of car trips seem to be generally longer than the corresponding reported TBI distance. This may be one reason the durations are greater in the rerouting analysis.

In [ ]:
pd.concat([df[df["mode"] == "Car"]["rerouting_distance_car"].describe(), df[df["mode"] == "Car"]["distance"].describe()], axis=1)

This scatter plot shows the difference in distances between observed and rerouted distances for observed car linked trips. In general, it seems that the rerouting distances seem to roughly match the actual distances for observed car trips, although it is slightly bottom heavy (actual distances are lower).

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))
sns.scatterplot(data=df[df["mode"] == "Car"], x="distance", y="rerouting_distance_car", ax=ax)
x_ref = np.linspace(0, 150)
plt.plot(x_ref, x_ref, color="r", linewidth=2)

The summary statsitics reflect the graph. Roughly 75% of all trips have a greater rerouting distance, with the other 25% having a greater observed distance. 

In [ ]:
(df[df["mode"] == "Car"]["distance"] - df[df["mode"] == "Car"]["rerouting_distance_car"]).describe(percentiles=[0.05, 0.1, 0.15, 0.2, 0.25, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95])

This scatter plot shows the observed and car-rerouted distances for non-car linked trips. This roughly reflects what was seen in the car linked trip graph above, which is reasonable as distance likely doesn't change drastically between a car and its corresponding non-car trip.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))
sns.scatterplot(data=df[df["mode"] != "Car"], x="distance", y="rerouting_distance_car", ax=ax)
x_ref = np.linspace(0, 80)
plt.plot(x_ref, x_ref, color="r", linewidth=2)

The summary statistics also roughly reflect what was seen with the car linked trips.

In [ ]:
(df[df["mode"] != "Car"]["distance"] - df[df["mode"] != "Car"]["rerouting_distance_car"]).describe(percentiles=[0.05, 0.1, 0.15, 0.2, 0.25, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95])

## Bike/Scooter Analysis

In [ ]:
bike_lts = pd.read_parquet(teams_dir + "/Data_processed/geodata/bike_lts.parquet")
bike_lts

In [ ]:
df["rerouting_duration_bike"] = (bike_lts.set_index("trip_id").loc[df["trip_id"]]["weight"] / 60).values # use weight instead of duration
df["rerouting_duration_bike_old"] = (bike_lts.set_index("trip_id").loc[df["trip_id"]]["duration_seconds"] / 60).values # use weight instead of duration
df["rerouting_distance_bike"] = (bike_lts.set_index("trip_id").loc[df["trip_id"]]["distance_meters"] * MILES_PER_METER).values

Going from the comparative summary statistics, it seems that rerouted durations for observed bike trips tend to be somewhat greater than the corresponding reported durations in the TBI data.

In [ ]:
pd.concat([df[df["mode"] == "Bike/Scooter"]["rerouting_duration_bike"].describe(), df[df["mode"] == "Bike/Scooter"]["duration"].describe()], axis=1)

This scatter plot shows the observed versus rerouted durations for observed bike trips. In general, most linked trips are above the y=x line, meaning that most linked trips have rerouted durations that are greater than their corresponding reported duration in the TBI data. There are also some trips that have absurd rerouted durations, which could be due to error with duration handling in the TBI cleaning process.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))
sns.scatterplot(data=df[df["mode"] == "Bike/Scooter"], x="duration", y="rerouting_duration_bike", ax=ax)
x_ref = np.linspace(0, 300)
plt.plot(x_ref, x_ref, color="r", linewidth=2)

This reflects what was seen in the above chart. About 90% of all observed bike trips have rerouted durations that were greater than their corresponding observed duration, and the central magnitudes for these duration differences are not insignificant (35 minutes for mean; 26 minutes for median).

In [ ]:
(df[df["mode"] == "Bike/Scooter"]["duration"] - df[df["mode"] == "Bike/Scooter"]["rerouting_duration_bike"]).describe(percentiles=[0.05, 0.1, 0.15, 0.2, 0.25, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95])

An analsyis of rerouted bike durations for non-bike trips is left out, as non-bike trips would include car trips, which would muddle the conclusions that could be drawn.

We will thus begin with bike/scooter distance analysis. In general, from the side by side summary statistics, it seems that rerouted distances tend to be greater than observed distances.

In [ ]:
pd.concat([df[df["mode"] == "Bike/Scooter"]["rerouting_distance_bike"].describe(), df[df["mode"] == "Bike/Scooter"]["distance"].describe()], axis=1)

This scatter plot shows observed/reported distances versus rerouted distances for observed bike trips. In general, despite the variance in the summary statistics, it seems that the observed and rerouted distances tend to match up, clustering around the y=x line, with the bottom being marginally heavier (observed distances are larger than rerouted distances).

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))
sns.scatterplot(data=df[df["mode"] == "Bike/Scooter"], x="distance", y="rerouting_distance_bike", ax=ax)
x_ref = np.linspace(0, 80)
plt.plot(x_ref, x_ref, color="r", linewidth=2)

Analyzing the summary statistics of the difference between observed and rerouted distances for observed bike trips, it seems that, like with bike durations, the rerouted distances seem to be generally larger than their observed counterparts (this accounts for about 80% of all bike linked trips). However, the magnitudes of these differences are not as large as with durations, with the mean/median differences having magnitudes of 0.5 miles. 

In [ ]:
(df[df["mode"] == "Bike/Scooter"]["distance"] - df[df["mode"] == "Bike/Scooter"]["rerouting_distance_bike"]).describe(percentiles=[0.05, 0.1, 0.15, 0.2, 0.25, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95])

## Walking analysis

In [ ]:
walk = pd.read_parquet(teams_dir + "/Data_processed/geodata/walk_trips_nogeom.parquet")
walk

In [ ]:
df["rerouting_duration_walk"] = (walk.set_index('trip_id').loc[df["trip_id"]]["duration_seconds"] / 60).values
df["rerouting_distance_walk"] = (walk.set_index('trip_id').loc[df["trip_id"]]["distance_meters"] * MILES_PER_METER).values

From the side by side comparison, it seems that the rerouting analysis seems to underestimate the duration of walk trips for observed walk trips. 

In [ ]:
pd.concat([df[df["mode"] == "Walk"]["rerouting_duration_walk"].describe(), df[df["mode"] == "Walk"]["duration"].describe()], axis=1)

This scatter plot shows the observed and rerouted durations for observed walk trips. In the scatter plot, there is a very large horizontal cluster for short rerouted duration along the entire duration axis, representing linked trips that had a greater duration than rerouted duration (indicating that rerouting underestimates duration). 

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))
sns.scatterplot(data=df[df["mode"] == "Walk"], x="duration", y="rerouting_duration_walk", ax=ax)
x_ref = np.linspace(0, 300)
plt.plot(x_ref, x_ref, color="r", linewidth=2)

It could be the case that this seemingly large difference between rerouted nad observed walking durations is due to the TBI data (the duration reported for walking trips are biased)--e.g., as seen below, trips taking longer than 100 minutes to complete have a median distance of 1.3 miles, which doesn't really make sense.

In [ ]:
df[(df["mode"] == "Walk") & (df["duration"] > 100)]["distance"].describe()

In [ ]:
# temp = pd.read_csv(teams_dir + "/data_processed/tbi_merged.csv")

# temp["rerouting_duration_walk"] = (walk.set_index('trip_id').loc[temp["trip_id"]]["duration_seconds"] / 60).values
# temp["rerouting_distance_walk"] = (walk.set_index('trip_id').loc[temp["trip_id"]]["distance_meters"] * MILES_PER_METER).values

# fig, ax = plt.subplots(figsize=(15, 10))
# sns.scatterplot(data=temp[temp["mode"] == "Walk"], x="duration", y="rerouting_duration_walk", ax=ax)
# x_ref = np.linspace(0, 300)
# plt.plot(x_ref, x_ref, color="r", linewidth=2)
# ax.set_xlim(0, 300)

Despite the strange pattern seen in the above scatter plot, it seems that the vast majority of the rerouted walk durations match roughly with the observed durations--80% of all walking trips have a absolute difference of about 10 minutes between the two. Therefore, despite the strange outlier cluster observed, it seems that rerouting duration for walking trips are roughly accurate.

In [ ]:
(df[df["mode"] == "Walk"]["duration"] - df[df["mode"] == "Walk"]["rerouting_duration_walk"]).describe(percentiles=[0.05, 0.1, 0.15, 0.2, 0.25, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95])

The rerouting analysis for distance of observed walking trips seem to be more or less in-line with the observed disatnce of these trips.

In [ ]:
pd.concat([df[df["mode"] == "Walk"]["rerouting_distance_walk"].describe(), df[df["mode"] == "Walk"]["distance"].describe()], axis=1)

This scatter plot shows the observed TBI/rerouted disatnces for observed walking trips. Unlike with durations, the two seem to eb roughly in line, although with the bottom of hte line being slightly heavier (rerouting underestimates distance of trips more times than not).

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))
sns.scatterplot(data=df[df["mode"] == "Walk"], x="distance", y="rerouting_distance_walk", ax=ax)
x_ref = np.linspace(0, 50)
plt.plot(x_ref, x_ref, color="r", linewidth=2)

The summary statistics of hte difference between observed/rerouted walking distance for observed walking trips mirros what was seen in the scatter plot and is relatively similar to the summary statistics for the duration difference for observed walking trips. Overall, the absolute idfference between the two is relatively low, with 85% of trips having an absolute difference of about 0.35 miles or less.

In [ ]:
(df[df["mode"] == "Walk"]["distance"] - df[df["mode"] == "Walk"]["rerouting_distance_walk"]).describe(percentiles=[0.05, 0.1, 0.15, 0.2, 0.25, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95])

## Transit analysis

This analysis is only for the trips that have a corresponding found transit trip in the rerouting analysis.

In [ ]:
transit = gpd.read_parquet(teams_dir + "/data_processed/geodata/transit_trips.parquet")

In [ ]:
# calculate duration from star/tend times
transit["start_time_dt"] = pd.to_datetime(transit["start_time"])
transit["end_time_dt"] = pd.to_datetime(transit["end_time"])

transit["duration"] = (transit["end_time_dt"] - transit["start_time_dt"]).apply(lambda x: x.seconds / 60)

In [ ]:
transit["num_transfers"] = (transit["leg_type"] == "TransitRouter.transfer") # temporary field to calculate number of transfers
transit["non_transit_duration"] = (transit["leg_type"].isin(["TransitRouter.access", "TransitRouter.egress"])) * (transit["duration"]) # non-transit time (access and egress)

In [ ]:
# project to US equidistant projection to calculate lengths of paths (in meters)
# https://spatialreference.org/ref/esri/usa-contiguous-equidistant-conic/
transit = transit.to_crs("ESRI:102005")

In [ ]:
transit["length"] = transit.length
transit["access_length"] = transit["length"]
transit

In [ ]:
# create a gdf for linked trips
agg_fns = {
    "trip_id": "first",
    "leg_index": "count",
    "start_time": "first",
    "end_time": "last",
    "origin_stop_id": "first", # throw away
    "origin_stop_name": "first",
    "destination_stop_id": "first",
    "destination_stop_name": "first",
    "route_id": "first", 
    "route_short_name": "first",
    "route_long_name": "first",
    "route_type": "first", 
    "leg_type": "first", # end throw away
    "start_time_dt": "first",
    "end_time_dt": "last",
    "duration": "sum",
    "length": "sum",
    "access_length": "first",
    "num_transfers": "sum",
    "non_transit_duration": "sum"
}

transit_grouped = transit.dissolve(by="trip_id", aggfunc=agg_fns) # merges geometries in addition to aggregating the rest of the columns

In [ ]:
transit_grouped

In [ ]:
transit_grouped.loc[-1] = -1
transit_grouped

In [ ]:
df["trip_id_adj"] = np.where(df["trip_id"].isin(transit_grouped.index), df["trip_id"], -1)
df["rerouting_duration_transit"] = transit_grouped.loc[df["trip_id_adj"]]["duration"].values
df["rerouting_distance_transit"] = transit_grouped.loc[df["trip_id_adj"]]["length"].values * MILES_PER_METER

From teh side by side summary statistics of rerouted/observed durations for transit trips with a corresponding rerouted transit trips, it seems that the rerouting analysis tends to overestimate the durations of transit trips.

In [ ]:
pd.concat([df[(df["mode"] == "Transit") & (df["rerouting_duration_transit"] >= 0)]["rerouting_duration_transit"].describe(), df[(df["mode"] == "Transit") & (df["rerouting_duration_transit"] >= 0)]["duration"].describe()], axis=1)

These scatter plots represent the observed/rerouted durations for observed transit trips that had a corresponding valid transit trip in the rerouting analysis. It overall seems that the rerouted durations are roughly equal to the observed durations, clustering around the y=x line. The scatter plot is a bit more botton-heavy, meaning that rerouting tends to underestimate transit durations. There is a peculiar hroizontal outlier that is characterized by a large and unreasonable rerouted duration compared to actual duration. 

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))
sns.scatterplot(data=df[(df["mode"] == "Transit") & (df["rerouting_duration_transit"] >= 0)], x="duration", y="rerouting_duration_transit", ax=ax)
x_ref = np.linspace(0, 350)
plt.plot(x_ref, x_ref, color="r", linewidth=2)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))
sns.scatterplot(data=df[(df["mode"] == "Transit") & (df["rerouting_duration_transit"] >= 0)], x="duration", y="rerouting_duration_transit", ax=ax)
x_ref = np.linspace(0, 350)
plt.plot(x_ref, x_ref, color="r", linewidth=2)
ax.set_ylim(0, 200)

As seen previuosly, the % of linked trips with rerouting duration greater/less than observed are roughly equal, but hte magnitudes for these are no insignificant, with 80% of trips having an aboslute difference of 18 minutes between the two.

In [ ]:
(df[(df["mode"] == "Transit") & (df["rerouting_duration_transit"] >= 0)]["duration"] - df[(df["mode"] == "Transit") & (df["rerouting_duration_transit"] >= 0)]["rerouting_duration_transit"]).describe(percentiles=[0.05, 0.1, 0.15, 0.2, 0.25, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95])

Comparing the summary statistics of rerouted and observed distances for observed transit trips that had a corresponding rerouted transit trip found for them, it seems that the rerouting analysis tend sto overestimate the distance of a trip. This could be due to the fact that transit trips in the TBI generally didn't seem to include the access/egress parts. 

In [ ]:
pd.concat([df[(df["mode"] == "Transit") & (df["rerouting_duration_transit"] >= 0)]["rerouting_distance_transit"].describe(), df[(df["mode"] == "Transit") & (df["rerouting_duration_transit"] >= 0)]["distance"].describe()], axis=1)

This scatter plot represents the rerouted/observed distances for observed transit trips that had a corresopnding rerouted transit trip found for them. In general, it seems that these two distances mirror each other roughly, clustering around y=x, although there is a small outlier cluster with high observed distance despite a low rerouted distance. 

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))
sns.scatterplot(data=df[(df["mode"] == "Transit") & (df["rerouting_duration_transit"] >= 0)], x="distance", y="rerouting_distance_transit", ax=ax)
x_ref = np.linspace(0, 50)
plt.plot(x_ref, x_ref, color="r", linewidth=2)
ax.set_ylim(0, 200)

The summary statsitics for the differences between observed/rerouted distances are more varied than those for durations. About 75% of transit linked trips have a larger observed distance, while the other 25% have a larger rerouted disatnce. The overlal magnitudes of difference seem to be relatively low, with 80% (15-95) of trips having an absolute difference of about 1.7 miles. 

In [ ]:
(df[(df["mode"] == "Transit") & (df["rerouting_duration_transit"] >= 0)]["distance"] - df[(df["mode"] == "Transit") & (df["rerouting_duration_transit"] >= 0)]["rerouting_distance_transit"]).describe(percentiles=[0.05, 0.1, 0.15, 0.2, 0.25, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95])

## Overall comparisons

### Comparison of summary statistics

The medians for durations from shortest to longest go like walk < car < bike < transit

In [ ]:
summary = pd.concat([df[df["mode"] == "Car"]["duration"].describe(), df[df["mode"] == "Bike/Scooter"]["duration"].describe(), df[df["mode"] == "Walk"]["duration"].describe(), df[df["mode"] == "Transit"]["duration"].describe()], axis=1)
summary.columns = ["Car", "Bike", "Walk", "Transit"]
summary

When looking at the relative order of the rerouted durations, the same order is observed, although while car and walk duration decrease with rerouting, transit and bike increase slightly.

In [ ]:
summary = pd.concat([df[df["mode"] == "Car"]["rerouting_duration_car"].describe(), df[df["mode"] == "Bike/Scooter"]["rerouting_duration_bike"].describe(), df[df["mode"] == "Walk"]["rerouting_duration_walk"].describe(), df[(df["mode"] == "Transit") & (df["rerouting_duration_transit"] >= 0)]["rerouting_duration_transit"].describe()], axis=1)
summary.columns = ["Car", "Bike", "Walk", "Transit"]
summary

The medians for observed distances from shortest to longest go walk, bike, car, and then transit. 

In [ ]:
summary = pd.concat([df[df["mode"] == "Car"]["distance"].describe(), df[df["mode"] == "Bike/Scooter"]["distance"].describe(), df[df["mode"] == "Walk"]["distance"].describe(), df[df["mode"] == "Transit"]["distance"].describe()], axis=1)
summary.columns = ["Car", "Bike", "Walk", "Transit"]
summary

The relative orders of distances for these modes when considering rerouting is the same, with car and bike median distance increasingly marginally and walk/transit decreasing marginally. 

In [ ]:
summary = pd.concat([df[df["mode"] == "Car"]["rerouting_distance_car"].describe(), df[df["mode"] == "Bike/Scooter"]["rerouting_distance_bike"].describe(), df[df["mode"] == "Walk"]["rerouting_distance_walk"].describe(), df[(df["mode"] == "Transit") & (df["rerouting_duration_transit"] >= 0)]["rerouting_distance_transit"].describe()], axis=1)
summary.columns = ["Car", "Bike", "Walk", "Transit"]
summary

When looking at the average speeds by modes (accounting for differences in distances w.r.t. duration), things make sense, with walk being the slowest, followed by biking, transit, and then car.

In [ ]:
summary = pd.concat([(df[df["mode"] == "Car"]["rerouting_distance_car"] / (df[df["mode"] == "Car"]["rerouting_duration_car"] + 1) * 60).describe(), (df[df["mode"] == "Bike/Scooter"]["rerouting_distance_bike"] / (df[df["mode"] == "Bike/Scooter"]["rerouting_duration_bike"] + 1) * 60).describe(), (df[df["mode"] == "Walk"]["rerouting_distance_walk"] / (df[df["mode"] == "Walk"]["rerouting_duration_walk"] + 1) * 60).describe(), (df[(df["mode"] == "Transit") & (df["rerouting_duration_transit"] >= 0)]["rerouting_distance_transit"] / (df[(df["mode"] == "Transit") & (df["rerouting_duration_transit"] >= 0)]["rerouting_duration_transit"] + 1) * 60).describe()], axis=1)
summary.columns = ["Car", "Bike", "Walk", "Transit"]
summary

### Intra-mode validation


When grouping by observed mode and looking at the rerouted durations for each one, the relative orderings of durations are still maintained. The only deviation from this is with hte observed walking trips, where biking is slightly slower than walking. 

This is only true with weights being used as the measure for duration; with the actual duration, biking remains the slowest among all of these groupings.

In [ ]:
pd.concat([df[df["mode"] == "Walk"]["rerouting_duration_walk"].describe(), df[df["mode"] == "Walk"]["rerouting_duration_bike"].describe(), df[df["mode"] == "Walk"]["rerouting_duration_car"].describe(), df[df["mode"] == "Walk"]["rerouting_duration_transit"].describe()], axis=1)

In [ ]:
pd.concat([df[df["mode"] == "Bike/Scooter"]["rerouting_duration_walk"].describe(), df[df["mode"] == "Bike/Scooter"]["rerouting_duration_bike"].describe(), df[df["mode"] == "Bike/Scooter"]["rerouting_duration_car"].describe(), df[df["mode"] == "Bike/Scooter"]["rerouting_duration_transit"].describe()], axis=1)

In [ ]:
pd.concat([df[df["mode"] == "Car"]["rerouting_duration_walk"].describe(), df[df["mode"] == "Car"]["rerouting_duration_bike"].describe(), df[df["mode"] == "Car"]["rerouting_duration_car"].describe(), df[df["mode"] == "Car"]["rerouting_duration_transit"].describe()], axis=1)

In [ ]:
pd.concat([df[df["mode"] == "Transit"]["rerouting_duration_walk"].describe(), df[df["mode"] == "Transit"]["rerouting_duration_bike"].describe(), df[df["mode"] == "Transit"]["rerouting_duration_car"].describe(), df[df["mode"] == "Transit"]["rerouting_duration_transit"].describe()], axis=1)

### Clustering analysis

Scatter plots to visualize clusters of observed vs rerouting by mode:

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))
sns.scatterplot(data=df[df["mode"] == "Car"], x="duration", y="rerouting_duration_car", ax=ax, label="car")
sns.scatterplot(data=df[df["mode"] == "Bike/Scooter"], x="duration", y="rerouting_duration_bike", ax=ax, label="bike")
sns.scatterplot(data=df[df["mode"] == "Walk"], x="duration", y="rerouting_duration_walk", ax=ax, label="walk")
sns.scatterplot(data=df[(df["mode"] == "Transit") & (df["rerouting_duration_transit"] >= 0)], x="duration", y="rerouting_duration_transit", ax=ax, label="transit")
x = np.linspace(0, 1000)
plt.plot(x, x, color="r")
plt.legend()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))
sns.scatterplot(data=df[df["mode"] == "Car"], x="distance", y="rerouting_distance_car", ax=ax, label="car")
sns.scatterplot(data=df[df["mode"] == "Bike/Scooter"], x="distance", y="rerouting_distance_bike", ax=ax, label="bike")
sns.scatterplot(data=df[df["mode"] == "Walk"], x="distance", y="rerouting_distance_walk", ax=ax, label="walk")
sns.scatterplot(data=df[(df["mode"] == "Transit") & (df["rerouting_duration_transit"] >= 0)], x="duration", y="rerouting_distance_transit", ax=ax, label="transit")
x = np.linspace(0, 400)
plt.plot(x, x, color="r")
plt.legend()